# GSB 5544 — Topic 4.1: Text as Data  
*Fill each `____` blank as you work; the ✅ checks ask for a sentence or two.*

## The road map

Week 3 asked *which rows of a table are most alike?* and answered it with a distance.
This week the observations are **emails** — there are no columns yet, only text. The whole
lesson is one journey, in three legs:

| Leg | Question | Sections | Where it lands in PA 4.1 |
|---|---|---|---|
| **Represent** | How does text become rows and columns? | 1 – 5 | parts 1 – 3 |
| **Weight** | Which columns deserve to count for more? (TF-IDF) | 6 | parts 4 – 5 |
| **Compare** | Given two rows of numbers, how similar are the emails? | 7 – 9 | parts 6 – 11 |

Every section ends with ✅ questions. Answer them *from the matrix in front of you* — the
point of this notebook is to keep looking at what the code produced, not just to run it.

In [ ]:
import pandas as pd
import numpy as np

---
## 1. Documents and corpora

Two words you will meet in every reading on this topic:

- A **document** is one unit of text — whatever we are treating as *one observation*: one email, one review, one tweet, one contract.
- A **corpus** is the collection of documents — the whole data set of text.

Our running example is four short emails (two from work, two that look like spam):

In [ ]:
corpus = pd.Series([
    "Send the invoice to Sam and the price to Kim",
    "The invoice and the price are attached",
    "Win your FREE prize now and claim the prize",
    "Free free FREE: click the link to win",
])
corpus

✅ **Fill in the blanks.**

1. Each element of `corpus` (for example `corpus[2]`) is one ____.
2. The whole Series `corpus` is the ____.
3. In PA 4.1, one document is one ____, and the corpus is the ____ column of the data frame.
4. In the table we are about to build, each document will become one ____ (row / column).

**Your answer:** *(write it here — replace this line)*

---
## 2. From text to tokens

Before we can count anything, the text has to be made **consistent** and then **cut into pieces**.

### 2a. Normalization — make the text consistent

Normalization means applying rules so that things that *should* count as the same actually
*look* the same. The most common rule is **lowercasing**: `Free`, `free`, and `FREE` are one
word, and we want them counted as one.

In [ ]:
corpus.str.____()

✅ In documents 2 and 3, which words were *different strings* before normalizing and are
*identical* after? How many distinct spellings collapsed into one?

**Your answer:** *(write it here — replace this line)*

### 2b. Tokenization — cut the text into units

Tokenization splits each document into **tokens** — the units we will count. Here a token is
a *word*: a run of letters or digits, with punctuation and spaces acting as the cuts.

In [ ]:
tokens = corpus.str.lower().str.____(r"\w+")     # \w+ = a run of word characters
tokens

In [ ]:
tokens.apply(len)                                       # how many tokens in each document

✅ **Document 0** has ____ tokens in total, but only ____ *distinct* tokens, because the
words ____ and ____ each appear twice.

**Your answer:** *(write it here — replace this line)*

### 2c. Tokens are words, not phrases

A token is a single word. A phrase such as *free prize* is **not** one token — it is two
adjacent tokens. If we want phrases to be counted, we count **combinations** of neighbouring
tokens: pairs are called **bigrams**, triples **trigrams**. (`CountVectorizer(ngram_range=(1, 2))`
would add every bigram as an extra column; the default counts single words only, and that is
what we use this week.)

✅ Write out the bigrams of the phrase *"click the link"*. How many are there?

**Your answer:** *(write it here — replace this line)*

---
## 3. Vocabulary and counting: the algorithm

Put the pieces in order and you have an **algorithm** — a fixed sequence of steps that turns
any corpus into a table:

| Step | Action | Result |
|---|---|---|
| 1 | **Normalize** each document (lowercase) | consistent text |
| 2 | **Tokenize** each document into words | a list of tokens per document |
| 3 | **Build the vocabulary** — the set of *distinct* tokens across the whole corpus | the **columns** of the table |
| 4 | **Count** how many times each vocabulary word appears in each document | one **row** per document |

The output is the **term-frequency (TF) matrix**, also called the document-term matrix:
documents down the side, vocabulary words across the top, counts in the cells.

- The **vocabulary** decides the columns — so the number of columns is the number of distinct words in the corpus.
- The **documents** decide the rows — one each.

✅ **Count by hand first.** Document 1 is *"The invoice and the price are attached"*.
Fill in its row of the table:

| `and` | `are` | `attached` | `invoice` | `price` | `the` | every other column |
|---|---|---|---|---|---|---|
| ____ | ____ | ____ | ____ | ____ | ____ | ____ |

And the whole matrix will have ____ rows and ____ columns. (Count the distinct words in
`tokens` above — or trust the code in Section 4 to tell you.)

**Your answer:** *(write it here — replace this line)*

---
## 4. The algorithm in code

`scikit-learn`'s `CountVectorizer` runs all four steps. We take it in pieces so that each
call can be matched to a step of the algorithm — and to the matrix it produces.

**Chunk 1 — create the vectorizer.** Nothing has touched the corpus yet; this object just
holds the settings (lowercase on, one-word tokens).

In [ ]:
from sklearn.feature_extraction.text import ____

vec = ____()
vec

**Chunk 2 — `fit`: learn the vocabulary.** `fit` reads the corpus, normalizes and tokenizes
every document, and collects the distinct words. It performs steps 1 – 3 and remembers the
vocabulary; it does **not** count anything yet.

In [ ]:
vec.____(corpus)
vocab = vec.____()
vocab, len(vocab)

✅ (a) `len(vocab)` is the number of ____ the matrix will have. (b) Is `vocab` in the order
the words appeared in the emails? (c) Which step of the algorithm has *not* happened yet?

**Your answer:** *(write it here — replace this line)*

**Chunk 3 — `transform`: count.** Now each document is counted against the learned vocabulary
— step 4. The result has one row per document and one column per vocabulary word.

In [ ]:
tf = vec.____(corpus)
tf.shape

✅ Read `tf.shape` as (____, ____). What decided the first number, and what decided the second?

**Your answer:** *(write it here — replace this line)*

**Chunk 4 — make it a data frame.** `tf` is stored as a *sparse* matrix (only the non-zero
cells are kept, because most cells are zero). `.toarray()` expands it, and the vocabulary
supplies the column names.

In [ ]:
tf_df = pd.DataFrame(tf.____(), columns=____)
tf_df

✅ Compare row 1 of `tf_df` with your hand count from Section 3. Do they agree — including
the 2 for `the`? Now check row 0 against your token count from Section 2b: which two columns
hold a 2?

**Your answer:** *(write it here — replace this line)*

**Chunk 5 — the shorthand.** `fit_transform` runs `fit` and then `transform` on the same
corpus in one call. It is what you will write in PA 4.1:

In [ ]:
tf_df2 = pd.DataFrame(vec.____(corpus).toarray(), columns=vec.get_feature_names_out())
tf_df2.equals(tf_df)          # same matrix as the two-step version

✅ When would you call `transform` **without** `fit`? (Hint: in PA 4.1 there are emails whose
spam status is unknown, and we will want to compare them with the known ones.)

**Your answer:** *(write it here — replace this line)*

---
## 5. Reading the TF matrix

The code is done; now interpret the object it made. Answer each question by looking at
`tf_df`, then check with code.

In [ ]:
tf_df.loc[____, ____]           # the cell for document 2, word "prize"

✅ (a) In words, what does the value in that cell mean? (b) Why is `tf_df.loc[3, "invoice"]`
zero? (c) What fraction of the whole matrix is zeros — and why is that fraction so high?

In [ ]:
(tf_df == 0).mean().mean()          # fraction of cells that are zero

**Your answer:** *(write it here — replace this line)*

In [ ]:
tf_df.____().sort_values(ascending=False).head(3)     # column totals: most-used words overall

✅ (d) `the` is the most-used word in the corpus. Does knowing an email contains `the` help you
tell the work emails (0, 1) from the spam-like emails (2, 3)? Which word in the top three
*does* help? (e) Document 0 and the shuffled sentence *"Kim the price and send to the invoice
to Sam"* — would they get the same row? What has the matrix thrown away?

**Your answer:** *(write it here — replace this line)*

---
## 6. TF-IDF, one ingredient at a time

Section 5 ended with a problem: the largest counts belong to words like `the`, `and`, `to` that
say nothing about what an email is *about*, while the words that do — `invoice`, `price`,
`prize` — have small counts. We fix it by building a **weight** for each word out of three
ingredients.

### 6a. Term frequency (TF)

This is the matrix we already have: TF(*document*, *word*) = how many times the word appears
in that document. It answers *"how much does **this** document use the word?"*

✅ Documents 0 and 1 share the word `the` (twice each) **and** the word `invoice` (once each).
Which of the two shared words is the stronger clue that they are about the same thing? Why
can't TF alone tell you?

**Your answer:** *(write it here — replace this line)*

### 6b. Document frequency (DF)

DF(*word*) = the number of **documents** that contain the word (not how many times). It is a
property of the *column*, not of a cell: one number per word.

In [ ]:
df = (tf_df ____).sum()            # True where the word appears; summing Trues counts documents
df.sort_values(ascending=False)

✅ Fill in: DF(`the`) = ____, DF(`and`) = ____, DF(`invoice`) = ____, DF(`price`) = ____,
DF(`prize`) = ____. Which words are common *across the corpus* and which are rare?

**Your answer:** *(write it here — replace this line)*

### 6c. Inverse document frequency (IDF)

We want a weight that is **large for rare words and small for common ones** — the opposite of
DF. The textbook definition is

$$\text{IDF}(\text{word}) = \log\!\left(\frac{N}{\text{DF}(\text{word})}\right), \qquad N = \text{number of documents.}$$

A word in every document has DF = N, so IDF = log(1) = **0**: its column is switched off.
A word in one document out of four has IDF = log(4) ≈ 1.39, the maximum.

In [ ]:
N = ____
idf_simple = np.log(____)
idf_simple.sort_values()

✅ (a) Which word has IDF exactly 0, and why? (b) Which words tie for the highest IDF, and what
do they have in common? (c) Put `the`, `and`, `invoice`, `prize` in order from smallest to
largest IDF.

**Your answer:** *(write it here — replace this line)*

### 6d. Put them together: TF × IDF

TF-IDF(*document*, *word*) = TF(*document*, *word*) × IDF(*word*). Each **cell** of the TF
matrix is multiplied by the weight of its **column**. A value is large only when the word is
used a lot in *this* document (TF) **and** is rare across the corpus (IDF).

In [ ]:
tfidf_manual = tf_df ____           # every column scaled by its own IDF
tfidf_manual.round(2)

✅ Look at **row 0** before (`tf_df`) and after (`tfidf_manual`). (a) `the` was the largest value
in the row (2). What is it now, and why? (b) `to` also had count 2. What happened to it, and why
is it different from `the`? (c) Which words in row 0 now score highest?

**Your answer:** *(write it here — replace this line)*

### 6e. `TfidfVectorizer` — the same idea, two small differences

`scikit-learn` builds TF-IDF in one call, exactly like `CountVectorizer`. Its numbers differ
from `tfidf_manual` for two practical reasons:

1. **Smoothed IDF**: it uses $\log\frac{1 + N}{1 + \text{DF}} + 1$ instead of $\log\frac{N}{\text{DF}}$, so no word is switched off completely (a word in every document gets weight 1, the minimum, rather than 0).
2. **Row normalization**: each row is rescaled to have length 1, so long documents and short documents are on the same footing — this will matter again in Section 7.

In [ ]:
from sklearn.feature_extraction.text import ____

tfidf_vec = ____()
tfidf_df = pd.DataFrame(tfidf_vec.____(corpus).toarray(),
                        columns=tfidf_vec.get_feature_names_out())

pd.DataFrame({"idf_simple": idf_simple, "idf_sklearn": tfidf_vec.idf_}, index=vocab).round(2).sort_values("idf_simple")

In [ ]:
tfidf_df.round(2)

✅ (a) Under `scikit-learn`'s IDF, `the` gets weight ____ (the smallest) instead of 0 — but is
the **ordering** of the words from least to most informative the same as in `idf_simple`?
(b) In **row 2**, TF said `prize` = 2 and `the` = 1. What does `tfidf_df` say for the same two
cells? (c) In one sentence: how do the values change when we move from TF to TF-IDF?

**Your answer:** *(write it here — replace this line)*

---
## 7. Comparing two documents: cosine similarity

Sections 1 – 6 built a representation: **one row of numbers per document**. Now the Week 3
question returns — *how alike are two rows?* — and we need a ruler.

**What the symbols mean.** Take document 0 and document 1:

- $x$ = the **row of document 0** in the matrix: 18 numbers, one per vocabulary word ($x_{\text{the}} = 2$, $x_{\text{invoice}} = 1$, $x_{\text{free}} = 0$, …). A row of numbers is a **vector**.
- $y$ = the row of document 1, the same 18 columns in the same order.

**The dot product** $x \cdot y = \sum_{\text{word}} x_{\text{word}} \, y_{\text{word}}$: multiply the two rows
column by column and add up. A word contributes only if **both** documents use it — a zero on
either side kills that term. So the dot product measures **how much vocabulary the two
documents share**, weighted by how heavily each uses it.

In [ ]:
x = tf_df.loc[____]
y = tf_df.loc[____]
(x * y)[(x * y) > 0]                 # only the words BOTH documents use survive

In [ ]:
dot = ____
dot

✅ (a) Which four words contribute to $x \cdot y$? (b) The dot product is 7 — how much of the 7
comes from the single word `the`, and is that the word you would *want* driving the answer?

**Your answer:** *(write it here — replace this line)*

**The lengths** $\lVert x \rVert = \sqrt{\sum_{\text{word}} x_{\text{word}}^2}$ and $\lVert y \rVert$
likewise: the size of each vector (Euclidean length, as in Week 3, measured from the origin).
A long email has bigger counts, so a bigger length **and** bigger dot products with everything
— length is exactly the thing we do *not* want the comparison to depend on.

**Cosine similarity** divides it out:

$$\text{cosine similarity}(x, y) = \frac{x \cdot y}{\lVert x \rVert \, \lVert y \rVert}$$

- the numerator rewards **shared vocabulary**,
- the denominator removes the effect of **document length**,
- what is left depends only on the *mix* of words — the direction of the vectors, not their size.

**Cosine distance** = 1 − cosine similarity, so that "small = similar", as in Week 3.

In [ ]:
len_x = np.sqrt(____)
len_y = np.sqrt(____)
cos_sim = dot / ____
len_x.round(3), len_y.round(3), cos_sim.round(3)

✅ **Fill in the blanks.**

1. $\lVert x \rVert = \sqrt{1+1+1+1+1+1+2^2+2^2} = \sqrt{\,____\,} \approx 3.74$ and $\lVert y \rVert = \sqrt{\,____\,} = 3$.
2. Cosine similarity of documents 0 and 1 = $7 / (3.74 \times 3) \approx$ ____.
3. For count vectors (no negatives), cosine similarity ranges from ____ (no shared words) to ____ (identical mix of words).
4. The cosine similarity of a document with **itself** is ____; the cosine *distance* of a document to itself is ____.

**Your answer:** *(write it here — replace this line)*

`scikit-learn` computes every pair at once:

In [ ]:
from sklearn.metrics.pairwise import ____

sim_tf = pd.DataFrame(____(tf_df), index=corpus.str[:22], columns=corpus.str[:22])
sim_tf.round(2)

✅ (a) Confirm the (0, 1) entry matches your hand calculation. (b) Which pair of documents is
**most** similar, and which is **least**? Read the sentences — does it make sense? (c) Why is
the diagonal all 1s?

**Your answer:** *(write it here — replace this line)*

**Why not Euclidean distance, as in Week 3?** Write document 3 out twice — same email, twice as
long. Every count doubles. Euclidean distance says it moved; cosine says it did not:

In [ ]:
doubled = tf_df.loc[[3]] * 2
euclid = np.sqrt(((doubled.values - tf_df.loc[[3]].values) ** 2).sum())
cosine_similarity(doubled, tf_df.loc[[3]])[0, 0].round(3), euclid.round(2)

✅ Which of the two rulers treats the doubled email as "the same email"? Why is that the right
behaviour for text?

**Your answer:** *(write it here — replace this line)*

---
## 8. How Section 6 and Section 7 fit together

They answer different questions, and it is worth saying so plainly:

| | Section 6 — TF-IDF | Section 7 — cosine similarity |
|---|---|---|
| **Question** | *What number goes in each cell?* | *How alike are two rows?* |
| **Acts on** | **one document at a time** (a row), using column-wide information (DF) | **two documents at a time** (a pair of rows) |
| **Output** | a **matrix** — the representation | a **number per pair** — the comparison |
| **Role** | decides what the vectors $x$ and $y$ *are* | decides how $x$ and $y$ are *compared* |

Section 7 builds on Section 6 because the vectors $x$ and $y$ **are rows of whichever matrix
you built** — feed it `tf_df` and you compare raw counts; feed it `tfidf_df` and you compare
weighted, length-normalized rows. Same ruler, different vectors, different similarities:

In [ ]:
sim_tfidf = pd.DataFrame(cosine_similarity(____), index=corpus.str[:22], columns=corpus.str[:22])
pd.concat({"cosine on TF": sim_tf.round(2), "cosine on TF-IDF": sim_tfidf.round(2)}, axis=1)

✅ (a) The similarity of documents 0 and 1 falls from 0.62 (TF) to 0.43 (TF-IDF). Using your
answer from Section 7 about the word `the`, explain why. (b) Does the **most similar pair**
change? (c) Complete the sentence: *"TF-IDF changes the ____ that go into the comparison;
cosine similarity is the ____ we compare them with."*

**Your answer:** *(write it here — replace this line)*

---
## 9. The payoff: classify an email by its nearest neighbours

Stack the three legs exactly the way PA 4.1 will:

1. Corpus → **TF or TF-IDF matrix** (Sections 3 – 6): text becomes rows of numbers.
2. A new email arrives with an **unknown** label — it is vectorized with the *same columns* (Section 4, chunk 5).
3. Compute the **cosine distance** from the new email to every email whose label is known (Section 7).
4. **Sort.** If the closest emails are mostly spam, bet spam.

That is nearest-neighbour classification, assembled from this notebook. A first look at the
real corpus (a sample of Enron email — the PA introduction tells the story):

In [ ]:
emails = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/refs/heads/main/data/enron_email.csv")
emails["spam"].value_counts(dropna=False)

1,980 emails labelled spam (1) or not (0) — and **10 with no label**, which PA 4.1 asks you to
guess. One practical warning: a few `body` entries are missing, and `CountVectorizer` refuses
`NaN` documents, so `.fillna("")` first.

In [ ]:
real_corpus = emails["body"].____
tf_real = CountVectorizer().fit_transform(real_corpus)
tf_real.shape

✅ Using Sections 3 – 4: what decided the first number in `tf_real.shape`, and what decided the
second? Roughly what fraction of that matrix do you expect to be zeros?

**Your answer:** *(write it here — replace this line)*

## The three lines to keep

| | |
|---|---|
| **Represent** | normalize → tokenize → vocabulary (columns) → count (rows) = the TF matrix; `CountVectorizer().fit_transform(corpus)` |
| **Weight** | TF-IDF = TF × IDF, IDF = log(N / DF): shrinks words that appear in most documents, keeps the distinctive ones; `TfidfVectorizer()` |
| **Compare** | cosine similarity = $x \cdot y \,/\, (\lVert x \rVert \lVert y \rVert)$ — shared vocabulary over document length; distance = 1 − similarity; sort and read the neighbours |

PA 4.1 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).